In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers.optimization import Adafactor
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import os

# ==========================================
# 1. IMPROVED CONFIGURATION
# ==========================================
MODEL_NAME = "OpenMed/OpenMed-NER-SpeciesDetect-BioMed-109M"
TRAIN_FILE = "species_525_cleaned.csv"
TEST_FILE = "Data_cleaned.csv"
OUTPUT_FILE = "results_openmed_v2_optimized.csv"

MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 5  # Increased slightly since we lowered LR

# ==========================================
# 2. ROBUST DATA PREPARATION
# ==========================================
def load_and_label_data(file_path, tokenizer):
    df = pd.read_csv(file_path)
    df['text'] = df['Title'].fillna('') + " " + df['Description'].fillna('')
    data = []

    print(f"Loading data from {file_path} ({len(df)} rows)...")

    for idx, row in df.iterrows():
        text = str(row['text'])
        species_raw = str(row['Species'])
        if species_raw == 'nan': continue

        # Clean up species list
        species_list = [s.strip() for s in species_raw.split(',') if s.strip()]

        # Create Character Masks (0=O, 1=B, 2=I)
        char_labels = np.zeros(len(text), dtype=int)
        found_any = False
        for species in species_list:
            start_idx = text.find(species)
            if start_idx != -1:
                found_any = True
                end_idx = start_idx + len(species)
                char_labels[start_idx] = 1
                char_labels[start_idx+1:end_idx] = 2

        if not found_any: continue

        # Tokenize with precise offset mapping
        try:
            tokenized = tokenizer(text, max_length=MAX_LEN, padding='max_length', truncation=True, return_offsets_mapping=True)
            labels = []
            for (start, end) in tokenized['offset_mapping']:
                if start == end:
                    labels.append(-100) # Special tokens (CLS, SEP, PAD)
                else:
                    # Assign label based on character mask
                    labels.append(char_labels[start])

            data.append({
                'input_ids': torch.tensor(tokenized['input_ids']),
                'attention_mask': torch.tensor(tokenized['attention_mask']),
                'labels': torch.tensor(labels)
            })
        except:
            continue

    return data

class NERDataset(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

# ==========================================
# 3. TRAINING WITH LOWER LR
# ==========================================
def train_experiment(train_loader, val_loader):
    print(f"Starting Training (LR=4e-5)...")

    # Reset head for 3 labels (O, B, I)
    model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=3, ignore_mismatched_sizes=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    # PARTIAL UNFREEZING (Head + Last 4 Layers)
    # This keeps the "medical knowledge" safe while learning "biodiversity"
    for param in model.parameters(): param.requires_grad = False

    # 1. Unfreeze Classifier
    for param in model.classifier.parameters(): param.requires_grad = True

    # 2. Unfreeze Last 4 Layers of Encoder
    # Auto-detect encoder type (BERT vs RoBERTa)
    if hasattr(model, 'bert'): encoder = model.bert.encoder
    elif hasattr(model, 'roberta'): encoder = model.roberta.encoder
    else: encoder = model.base_model.encoder

    for layer in encoder.layer[-4:]:
        for param in layer.parameters(): param.requires_grad = True

    # OPTIMIZER: ADAFACTOR WITH LOWER LR
    # We lower LR from 1e-3 to 4e-5 to prevent "brain damage" to the model
    optimizer = Adafactor(
        model.parameters(),
        lr=4e-5,  # <--- CRITICAL CHANGE
        eps=(1e-30, 1e-3),
        clip_threshold=1.0,
        decay_rate=-0.8,
        beta1=None,
        weight_decay=0.01, # Added weight decay for stability
        relative_step=False,
        scale_parameter=False,
        warmup_init=False,
    )

    model.train()
    for epoch in range(EPOCHS):
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

    return model

# ==========================================
# 4. SMART INFERENCE (Fixes ## artifacts)
# ==========================================
def extract_species(text, model, tokenizer):
    device = next(model.parameters()).device
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=2)[0].cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].cpu())

    extracted = []
    current_ent = []

    for token, label in zip(tokens, predictions):
        if token in tokenizer.all_special_tokens: continue

        # LOGIC: If it's a subword (starts with ##), force-attach it to previous word
        if token.startswith("##"):
            if current_ent: current_ent.append(token)
            continue

        if label == 1: # B-SPECIES
            if current_ent: extracted.append(tokenizer.convert_tokens_to_string(current_ent))
            current_ent = [token]
        elif label == 2: # I-SPECIES
            if current_ent: current_ent.append(token)
            else: current_ent = [token]
        else: # O
            if current_ent:
                extracted.append(tokenizer.convert_tokens_to_string(current_ent))
                current_ent = []

    if current_ent: extracted.append(tokenizer.convert_tokens_to_string(current_ent))

    # Clean Final List
    clean_list = []
    for s in extracted:
        # Remove artifacts
        s = s.replace(" ##", "").replace("##", "").strip()
        # Filter out noise (single letters)
        if len(s) > 2:
            clean_list.append(s)

    return list(set(clean_list))

def calculate_row_metrics_robust(row):
    # Robust metric that ignores Case and Spaces
    gt_str = str(row['Ground_Truth_List'])
    pred_str = str(row['Extracted_List'])

    # 1. Parse Ground Truth
    if gt_str == 'nan': y_true = set()
    else: y_true = set([s.strip().lower() for s in row['Ground_Truth_List']])

    # 2. Parse Predictions
    y_pred = set([s.strip().lower() for s in row['Extracted_List']])

    # 3. Fuzzy Match
    tp = 0
    matched_true = set()
    for p in y_pred:
        for t in y_true:
            if t in matched_true: continue
            # Match if exact OR substring (e.g. "ostrea" matches "ostrea edulis")
            if p == t or (len(p)>3 and p in t) or (len(t)>3 and t in p):
                tp += 1
                matched_true.add(t)
                break

    fp = len(y_pred) - tp
    fn = len(y_true) - tp # use TP count for matched truths

    # Avoid div by zero
    p = tp / (tp + fp) if (tp + fp) > 0 else (1.0 if not y_pred else 0.0)
    r = tp / (tp + fn) if (tp + fn) > 0 else 1.0
    f1 = 2 * (p*r) / (p+r) if (p+r) > 0 else 0.0

    return pd.Series([p, r, f1])

# ==========================================
# 5. MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, add_prefix_space=True)

    # 1. TRAIN
    processed_data = load_and_label_data(TRAIN_FILE, tokenizer)
    train_data, val_data = train_test_split(processed_data, test_size=0.1, random_state=42)

    train_loader = DataLoader(NERDataset(train_data), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(NERDataset(val_data), batch_size=BATCH_SIZE)

    # We just run one optimized experiment now
    model = train_experiment(train_loader, val_loader)

    # 2. TEST
    print(f"\nEvaluating on {TEST_FILE}...")
    df_test = pd.read_csv(TEST_FILE)
    df_test['text'] = df_test['Title'].fillna('') + " " + df_test['Description'].fillna('')

    if 'Species' in df_test.columns:
        df_test['Ground_Truth_List'] = df_test['Species'].apply(lambda x: [s.strip() for s in str(x).split(',')] if str(x).lower() != 'nan' else [])
    else:
        df_test['Ground_Truth_List'] = [[] for _ in range(len(df_test))]

    model.eval()
    tqdm.pandas(desc="Extracting")
    df_test['Extracted_List'] = df_test['text'].progress_apply(lambda x: extract_species(x, model, tokenizer))

    # Calculate Metrics
    metrics = df_test.apply(calculate_row_metrics_robust, axis=1)
    metrics.columns = ['Precision', 'Recall', 'F1']
    df_test = pd.concat([df_test, metrics], axis=1)

    # 3. PRINT REPORT
    print("\n" + "="*40)
    print("FINAL ROBUST PERFORMANCE (V2)")
    print("="*40)
    print(f"Macro Precision: {df_test['Precision'].mean():.4f}")
    print(f"Macro Recall:    {df_test['Recall'].mean():.4f}")
    print(f"Macro F1 Score:  {df_test['F1'].mean():.4f}")

    # Global
    total_tp, total_fp, total_fn = 0, 0, 0
    for i, row in df_test.iterrows():
        # Re-run robust matching logic for global counts
        y_true = set([s.strip().lower() for s in row['Ground_Truth_List']])
        y_pred = set([s.strip().lower() for s in row['Extracted_List']])

        tp = 0
        matched = set()
        for p in y_pred:
            for t in y_true:
                if t in matched: continue
                if p == t or (len(p)>3 and p in t) or (len(t)>3 and t in p):
                    tp += 1
                    matched.add(t)
                    break
        total_tp += tp
        total_fp += (len(y_pred) - tp)
        total_fn += (len(y_true) - tp)

    g_p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    g_r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    g_f1 = 2 * (g_p * g_r) / (g_p + g_r) if (g_p + g_r) > 0 else 0

    print("-" * 30)
    print(f"Total Species:      {total_tp + total_fn}")
    print(f"Correctly Found:    {total_tp}")
    print(f"Missed:             {total_fn}")
    print("-" * 30)
    print(f"Global Precision:   {g_p:.4f}")
    print(f"Global Recall:      {g_r:.4f}")
    print(f"Global F1 Score:    {g_f1:.4f}")
    print("="*40)

    # Save
    df_test['Extracted'] = df_test['Extracted_List'].apply(lambda x: ", ".join(x))
    df_test['Ground Truth'] = df_test['Ground_Truth_List'].apply(lambda x: ", ".join(x))
    cols = [c for c in df_test.columns if c not in ['Ground_Truth_List', 'Extracted_List']]
    df_test[cols].to_csv(OUTPUT_FILE, index=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/750 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Epoch 4 Loss: 0.0659


Epoch 5:   0%|          | 0/23 [00:00<?, ?it/s]

Epoch 5 Loss: 0.0600

Evaluating on Data_cleaned.csv...


Extracting:   0%|          | 0/44 [00:00<?, ?it/s]


FINAL ROBUST PERFORMANCE (V2)
Macro Precision: 0.8633
Macro Recall:    0.9421
Macro F1 Score:  0.8791
------------------------------
Total Species:      63
Correctly Found:    56
Missed:             7
------------------------------
Global Precision:   0.8116
Global Recall:      0.8889
Global F1 Score:    0.8485
